# fp16 training step with dynamic loss scaling

Mixed-precision training keeps float32 **master** weights but runs the forward and backward pass in float16. float16 tops out at 65504 and flushes values below about $6 \times 10^{-8}$ to zero, so small gradients vanish. **Loss scaling** multiplies the loss by a large factor $S$ before differentiating (shifting gradients into fp16's range) and divides the gradients by $S$ afterwards. **Dynamic** scaling lowers $S$ when gradients overflow and raises it after a run of good steps.

Model: linear regression, `params = {'w': (D,), 'b': ()}` in float32, loss $L = \mathrm{mean}\bigl((x w + b - y)^2\bigr)$.

Write `scaled_step(params, x, y, lr, scale, good_steps, growth_interval)`:

1. Cast `params`, `x` (`(N, D)`) and `y` (`(N,)`) to **float16**. Compute the scaled loss $S \cdot L$ in float16 (cast `scale` to float16 too) and its gradient with respect to the float16 parameters.
2. Cast the gradients to float32 and divide by `scale`.
3. If every gradient value is finite:
   - $p \leftarrow p - \eta\, g$ on the float32 master weights;
   - `good_steps += 1`; if it reaches `growth_interval`, set `scale *= 2` and `good_steps = 0`.
4. Otherwise (overflow): leave the parameters unchanged, `scale /= 2`, `good_steps = 0`.

Return `(new_params, new_scale, new_good_steps, finite)`: a float32 dict, a 0-d float32, a 0-d int and a 0-d bool. Use `jnp.where`, not Python `if`, on array values.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime. Results are compared with a loose tolerance because float16 rounding can differ slightly between JAX versions.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def scaled_step(params, x, y, lr, scale, good_steps, growth_interval):
    """Return (new_params f32, new_scale, new_good_steps, finite)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [1.0, 0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[1.0, 2.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 1.0], "dtype": "float64"}, 0.1, 1024.0, 0, 100], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.0, "dtype": "float32"}], ["w", {"__nd__": [0.8999999761581421, -0.10000000149011612], "dtype": "float32"}]]}, {"__nd__": 1024.0, "dtype": "float32"}, {"__nd__": 1, "dtype": "int32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [1.0, 0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[1.0, 2.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 1.0], "dtype": "float64"}, 0.1, 65536.0, 5, 100], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.0, "dtype": "float32"}], ["w", {"__nd__": [1.0, 0.0], "dtype": "float32"}]]}, {"__nd__": 32768.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int32"}, {"__nd__": false, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.35966845567139044, 1.2036751014877107, 1.3968681196180939, 0.31723592758455404], "dtype": "float64"}], ["b", {"__nd__": 0.2, "dtype": "float64"}]]}, {"__nd__": [[0.41413419203577684, -0.4895498035052234, -0.9141207048308895, -0.9003606290309881], [-0.9984312679885344, 0.9292048240939573, -0.05629191668093178, 0.1282790949733999], [-0.6399704392204456, -1.0878172085644096, -1.2019541755133116, -0.8415691844772564], [0.5991567147195185, 0.01844008196322814, -0.45677163924420827, -0.23927840186540572], [-1.427335350087018, 1.2306824377006564, -1.2164154219415413, 0.04238178066477212], [2.1369751648960755, -2.5511902614867554, -1.406959579319153, -0.7237059839987247], [0.1165516397529902, -1.7148763387690875, -0.23527948055323214, -0.028427676959011872], [0.1705221900247991, -2.3883732943063247, 0.6464356261864594, 1.5968682223798234]], "dtype": "float64"}, {"__nd__": [0.4365991726349167, -0.7231163455427266, -0.6134333623522121, -2.646308228460525, 0.4760024616261362, 1.5030674802640775, 0.650162096489897, -2.9772645036295913], "dtype": "float64"}, 0.05, 256.0, 9, 10], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.3125976622104645, "dtype": "float32"}], ["w", {"__nd__": [-0.1518559604883194, 0.8827767372131348, 1.1968680620193481, 0.1670406013727188], "dtype": "float32"}]]}, {"__nd__": 512.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.42597678713161496, -0.10285642667386714, -0.48352906075718616, 1.0332246946582169], "dtype": "float64"}], ["b", {"__nd__": 0.2, "dtype": "float64"}]]}, {"__nd__": [[-34.88569216911064, -14.355115589191572, 11.911505597543249, 19.818565262952305], [3.3828555917427887, 21.105578488677352, 10.403738896063528, -21.18535922300838], [25.87038940285961, -10.113355736669643, -32.61144865400921, 8.760720173926927], [36.28543790755193, 19.183454241304542, 19.946650506809224, 3.8127723281558845], [-49.30009604884677, 13.600572941296274, 3.1505562846126085, -52.82252739339165], [-13.0370540381319, -17.593693299826864, 21.22253294248903, 2.4675295621140645], [5.964437313954708, 4.854312418766565, -26.561114518104624, -39.27644989186155], [1.1433333028090666, -5.867601155184009, -0.28501504186592846, -3.327411704305163]], "dtype": "float64"}, {"__nd__": [-9.98434237352879, 11.103738121817903, 3.8367921022265703, -12.457678095385083, -37.99208181572714, -51.80055100412363, -28.81417019196863, -3.0649128022300234], "dtype": "float64"}, 0.01, 32768.0, 3, 10], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.20000000298023224, "dtype": "float32"}], ["w", {"__nd__": [-0.42597678303718567, -0.10285642743110657, -0.48352906107902527, 1.0332247018814087], "dtype": "float32"}]]}, {"__nd__": 16384.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int32"}, {"__nd__": false, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.91128919821284, -0.41996615491067374, 0.6772214184157], "dtype": "float64"}], ["b", {"__nd__": 0.2, "dtype": "float64"}]]}, {"__nd__": [[0.0011607997643442935, 0.0023799673928153304, 0.0002870668892446683], [0.0014944175592675353, 0.00023437223540164867, -0.0012386449327880175], [-0.002270643234553666, 0.0002972390620755206, 0.0008884705315099355], [0.002469564213913771, -0.000668264916757847, 0.0004901525821400374], [-0.00011585572773690406, -0.0023684030639767022, 0.0003748061805284154], [-0.0003217578118801296, -0.0012764107006043148, 0.001687811262339238], [-0.0006170702111319552, 0.00046888332055268077, -0.0002654266950246605], [-0.0007165401490616276, -0.00016788328716767616, 0.0007930627385336955], [-0.0006899549395889609, -0.0016514766493901537, 0.001796372147745623], [0.00043936429679943044, -0.0009590377267974494, -9.169039592660274e-05], [-0.0015647731862893201, -6.519451830897889e-05, 0.0008606304210250871], [-0.0019317392163238367, 0.0008670360218560247, 0.0008076404702564309], [-0.00027186961408035123, 0.00010247751609403443, -0.00033343488387829984], [-0.00044591067331802627, -0.0018617889790492365, 0.0016304031003557449], [-0.0006714792014687927, 0.00013477256648373215, -0.0002278488487858825], [0.0004554296213753734, 0.000567103241658828, -0.0010212312225774307]], "dtype": "float64"}, {"__nd__": [0.000362291064996665, -0.0002699760852090315, -0.0008835347648185112, 0.0012970279935412097, 0.0011785617445832632, -0.0009043964522532456, -0.0003022400320613513, -0.0010107654404609416, -0.0011373779553183428, -0.0009824123758251028, 0.0005546681019487977, -0.0014165852837098297, -0.0014626623662112427, 0.0002875983947485101, -0.0007196332336580128, 0.0012292373915639666], "dtype": "float64"}, 0.1, 32768.0, 0, 2000], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.159912109375, "dtype": "float32"}], ["w", {"__nd__": [0.9112981557846069, -0.4199560880661011, 0.6772052049636841], "dtype": "float32"}]]}, {"__nd__": 32768.0, "dtype": "float32"}, {"__nd__": 1, "dtype": "int32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.9608268373259189, -0.48623423678366556, -0.7756195816707919, -2.3148474428126415, 0.34784524628557817], "dtype": "float64"}], ["b", {"__nd__": 0.2, "dtype": "float64"}]]}, {"__nd__": [[0.6952167214422953, 0.955122704987263, -0.18845220196397422, 0.6678324374127321, -0.15451560877510476], [-0.09423383843573262, -0.8017210232205202, -0.15790547771500701, -0.9434172675695394, 0.59967054185928], [-0.6749887881664435, -1.0456958460685555, -0.1077531253826381, -0.0588423847241854, -1.6897742145532428], [-1.0706458982850466, -0.5858370310795251, 0.3810604752453028, 0.5208107175862352, 0.7826648363488281]], "dtype": "float64"}, {"__nd__": [-0.2666647694348976, -0.8366388970125821, -1.4405831890807592, 0.09864790892963123], "dtype": "float64"}, 0.1, 1.0, 1, 2], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": -0.0263671875, "dtype": "float32"}], ["w", {"__nd__": [-0.7761588096618652, -0.09092172980308533, -0.7542206645011902, -2.0515661239624023, 0.4085874557495117], "dtype": "float32"}]]}, {"__nd__": 2.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int32"}, {"__nd__": true, "dtype": "bool"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.4187404019054387, 0.4152003349305387], "dtype": "float64"}], ["b", {"__nd__": 0.2, "dtype": "float64"}]]}, {"__nd__": [[300.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, 0.1, 8.0, 7, 8], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.20000000298023224, "dtype": "float32"}], ["w", {"__nd__": [-0.4187403917312622, 0.4152003228664398], "dtype": "float32"}]]}, {"__nd__": 4.0, "dtype": "float32"}, {"__nd__": 0, "dtype": "int32"}, {"__nd__": false, "dtype": "bool"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(scaled_step, json.loads(_TESTS), 'jax', *[0.002, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")